In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import plotly.express as px
from src.score import Score
import os 
import os
from termcolor import colored

from src.score import Score
from src.log_data import *

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error


from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

current_dir = os.getcwd()
con = sqlite3.connect("data/main_database.db")
pd.set_option('display.float_format', lambda x: '%.3f' % x)
pd.set_option('display.max_columns', None)

In [ ]:
query = ''' 
select
    * 

from summary_per_game
where 1=1 
    and sentence is not null
    and wpm is not null
    
''' 

df_text = pd.read_sql_query(query, con)
text = ' '.join(df_text['sentence'].tolist())



def get_tokens(corpus):
    # Initialize a BPE tokenizer
    tokenizer = Tokenizer(BPE())

    # Customize pre-tokenization to split on whitespace
    tokenizer.pre_tokenizer = Whitespace()

    # Train the tokenizer on the input text
    trainer = BpeTrainer(vocab_size=10000, min_frequency=2)
    tokenizer.train_from_iterator([corpus], trainer=trainer)

    # Encode the input text using the trained tokenizer
    encoding = tokenizer.encode(corpus)

    # Extract the subword tokens from the encoding
    tokens = encoding.tokens

    return tokens


tokens = get_tokens(text)

In [ ]:
len(tokens), len(set(tokens)), len(text), len(text.split(' ')), len(set(text.split(' ')))

In [ ]:
df_text.head()

In [ ]:
df_text = df_text[['sentence', 'wpm', 'keys_to_press', 'train', 'train_easy', 'capitalized_words', 'capitalized_letters']].copy()

for token in set(tokens):
    df_text[token] = df_text['sentence'].apply(lambda x: x.count(token))

In [ ]:
df_text

In [ ]:
X = df_text.drop(['sentence', 'wpm'], axis=1)
y = df_text['wpm']


# Assume X and y are your features and target variable, respectively
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Initialize a Random Forest model
model = RandomForestRegressor(n_estimators=100, random_state=42)

# Fit the model to the training data
model.fit(X_train, y_train)

# Use the model to make predictions on the testing data
y_pred = model.predict(X_test)

# Print a regression report to evaluate the model's performance
print("R-squared score:", r2_score(y_test, y_pred))
print("Mean squared error:", mean_squared_error(y_test, y_pred))